# MVP de Engenharia de Dados — Etapa 5b: Solução do problema

---

> "Chegou o momento de se solucionar o problema em questão, definido preliminarmente nos objetivos. Deve-se buscar respostas para as perguntas elencadas. Para cada resposta obtida tecnicamente através da análise dos dados deve haver uma discussão do seu resultado, conectando os números obtidos às respostas ao problema a ser solucionado."

Cada seção abaixo responde a uma das oito perguntas declaradas em [`docs/01-objetivo.md`](../docs/01-objetivo.md), **antes** de qualquer coleta. Cada uma traz a consulta, o gráfico e a discussão do que o número significa.

As consultas usam as visões multidimensionais criadas em [`sql/40_views_analiticas.sql`](../sql/40_views_analiticas.sql), e estão também versionadas em [`sql/45_perguntas_negocio.sql`](../sql/45_perguntas_negocio.sql). Sobre elas são aplicados os operadores analíticos da Aula 1: *slice and dice* (filtros por natureza e por ano), *roll-up* e *drill-down* (navegação entre categoria e natureza, entre período e hora), e *pivoting* (troca da perspectiva de agrupamento).

> **Sobre os números apresentados.** Referem-se à coleta de agosto de 2026. A SSP-SP republica os arquivos periodicamente, de modo que uma nova execução do pipeline pode produzir números ligeiramente diferentes.

In [ ]:
PROJETO_ID = "mvp-criminalidade-sorocaba"
REPO_DIR   = "/content/mvp-engenharia-dados"
REPO_URL   = "https://github.com/SEU_USUARIO/mvp-engenharia-dados.git"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import os

cliente_bq = bigquery.Client(project=PROJETO_ID)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

def consultar(sql):
    return cliente_bq.query(sql.replace("@projeto", PROJETO_ID)).to_dataframe()

# Cria as visões multidimensionais sobre o esquema estrela
with open(f"{REPO_DIR}/sql/40_views_analiticas.sql", encoding="utf-8") as arquivo:
    cliente_bq.query(arquivo.read().replace("@projeto", PROJETO_ID)).result()
print("visões analíticas criadas")

---
## P1. Como evoluiu o volume de ocorrências, em absoluto e em taxa por 100 mil habitantes?

In [ ]:
p1 = consultar("""
SELECT
  ano,
  SUM(ocorrencias)                                     AS ocorrencias,
  MAX(populacao)                                       AS populacao,
  MAX(origem_populacao)                                AS origem_populacao,
  ROUND(100000 * SUM(ocorrencias) / MAX(populacao), 1) AS taxa_por_100_mil
FROM `@projeto.dw.vw_taxa_anual`
GROUP BY ano
ORDER BY ano
""")
p1["variacao_absoluta_%"] = (p1.ocorrencias.pct_change() * 100).round(1)
p1["variacao_taxa_%"] = (p1.taxa_por_100_mil.pct_change() * 100).round(1)
display(p1)

figura, (esq, dir) = plt.subplots(1, 2, figsize=(12, 4))
esq.bar(p1.ano.astype(str), p1.ocorrencias, color="#34495e")
esq.set_title("Ocorrências registradas (absoluto)")
for x, y in zip(p1.ano.astype(str), p1.ocorrencias):
    esq.text(x, y, f"{y:,}".replace(",", "."), ha="center", va="bottom", fontsize=9)
dir.plot(p1.ano.astype(str), p1.taxa_por_100_mil, marker="o", color="#c0392b", linewidth=2)
dir.set_title("Taxa por 100 mil habitantes")
for x, y in zip(p1.ano.astype(str), p1.taxa_por_100_mil):
    dir.text(x, y, f"{y:,.0f}", ha="center", va="bottom", fontsize=9)
dir.set_ylim(p1.taxa_por_100_mil.min() * 0.95, p1.taxa_por_100_mil.max() * 1.05)
plt.tight_layout(); plt.show()

### Discussão

**Resposta:** entre 2022 e 2025, as ocorrências registradas em Sorocaba passaram de **15.212 para 16.806 — um aumento de 10,5%**. Mas a taxa por 100 mil habitantes subiu de **2.102 para 2.205, apenas 4,9%**.

**A diferença entre os dois números é a resposta mais importante desta pergunta.** Metade do crescimento aparente não é aumento de criminalidade: é aumento de população. Sorocaba ganhou cerca de 38 mil habitantes no período, e uma cidade maior produz mais ocorrências pelo simples fato de ter mais gente. Sem o cruzamento com os dados do IBGE — o segundo fato do modelo dimensional — a leitura de "10% a mais de crime" seria defensável e estaria errada.

**O formato da série também importa.** O crescimento não é uma tendência de alta contínua: a taxa sobe forte em 2023 (+8,8%), cai em 2024 (-5,6%) e volta a subir em 2025 (+2,1%). O ano de 2023 é o pico do período, não 2025. Uma leitura que comparasse apenas as pontas concluiria "alta constante"; a série mostra oscilação em torno de um patamar, com um ano atípico.

**Ressalva.** Como declarado no objetivo, a base mede ocorrências **comunicadas**, não crimes ocorridos. Parte do crescimento pode refletir maior propensão a registrar — e o período coincide com a consolidação da Delegacia Eletrônica, que responde pela maioria dos registros e reduz drasticamente o custo de fazer um boletim. Não é possível, com esta base, separar os dois efeitos.

---
## P2. Quais naturezas concentram o maior volume, e quais mais cresceram ou caíram?

In [ ]:
# Roll-up: subtotais por categoria e total geral com ROLLUP
p2_categoria = consultar("""
SELECT
  IFNULL(categoria, 'TOTAL GERAL') AS categoria,
  SUM(qtd_ocorrencia)              AS ocorrencias,
  ROUND(100 * SUM(qtd_ocorrencia) / SUM(SUM(qtd_ocorrencia)) OVER (), 1) AS percentual
FROM `@projeto.dw.vw_ocorrencias`
WHERE ano_estatistica BETWEEN 2022 AND 2025
GROUP BY ROLLUP (categoria)
ORDER BY ocorrencias DESC
""")
display(p2_categoria)

# Drill-down: da categoria para a natureza
p2_natureza = consultar("""
SELECT categoria, natureza_apurada, SUM(qtd_ocorrencia) AS ocorrencias
FROM `@projeto.dw.vw_ocorrencias`
WHERE ano_estatistica BETWEEN 2022 AND 2025
GROUP BY categoria, natureza_apurada
ORDER BY ocorrencias DESC
LIMIT 10
""")
display(p2_natureza)

figura, eixo = plt.subplots(figsize=(9, 4))
dados = p2_natureza.iloc[::-1]
eixo.barh(dados.natureza_apurada, dados.ocorrencias, color="#2c3e50")
eixo.set_title("Dez naturezas criminais mais frequentes (2022-2025)")
plt.tight_layout(); plt.show()

In [ ]:
p2_variacao = consultar("""
SELECT
  natureza_apurada,
  categoria,
  SUM(IF(ano = 2022, ocorrencias, 0)) AS em_2022,
  SUM(IF(ano = 2025, ocorrencias, 0)) AS em_2025,
  ROUND(SUM(IF(ano = 2022, taxa_por_100_mil, 0)), 1) AS taxa_2022,
  ROUND(SUM(IF(ano = 2025, taxa_por_100_mil, 0)), 1) AS taxa_2025,
  ROUND(100 * SAFE_DIVIDE(
    SUM(IF(ano = 2025, ocorrencias, 0)) - SUM(IF(ano = 2022, ocorrencias, 0)),
    SUM(IF(ano = 2022, ocorrencias, 0))), 1) AS variacao_percentual
FROM `@projeto.dw.vw_taxa_anual`
WHERE ano IN (2022, 2025)
GROUP BY natureza_apurada, categoria
HAVING em_2022 + em_2025 >= 100
ORDER BY variacao_percentual DESC
""")
display(p2_variacao)

figura, eixo = plt.subplots(figsize=(9, 5))
cores = ["#c0392b" if v > 0 else "#27ae60" for v in p2_variacao.variacao_percentual[::-1]]
eixo.barh(p2_variacao.natureza_apurada[::-1], p2_variacao.variacao_percentual[::-1], color=cores)
eixo.axvline(0, color="black", linewidth=0.8)
eixo.set_xlabel("variação % de 2022 para 2025")
eixo.set_title("Quem cresceu e quem caiu")
plt.tight_layout(); plt.show()

### Discussão

**Resposta:** a criminalidade registrada em Sorocaba é, em esmagadora maioria, **crime contra o patrimônio: 77,6%** do total entre 2022 e 2025. Crimes contra a pessoa são 13,1%, trânsito 4,9%, drogas e armas 2,8% e dignidade sexual 1,6%. Uma única natureza — `FURTO - OUTROS` — responde por **57,3% de tudo**.

**O movimento interno contradiz o total.** Enquanto o volume geral subiu 4,9% em taxa, as naturezas se moveram em direções opostas:

| Em alta | | Em queda | |
|---|---|---|---|
| Tentativa de homicídio | +71,8% | Roubo — outros | **-33,3%** |
| Lesão corporal dolosa | +43,6% | Roubo de veículo | **-23,9%** |
| Furto de veículo | +36,7% | Tráfico de entorpecentes | -3,7% |
| Lesão corporal culposa (trânsito) | +30,1% | | |
| Estupro de vulnerável | +23,6% | | |

**O achado mais relevante é a troca entre roubo e furto.** O roubo — crime com violência ou grave ameaça — caiu um terço, enquanto o furto de veículo subiu 37%. Juridicamente, a diferença entre os dois é exatamente a presença de violência. A leitura possível é que a cidade registrou **menos crimes violentos contra o patrimônio e mais crimes patrimoniais sem confronto**, o que é uma melhora do ponto de vista do risco à integridade física, ainda que o prejuízo material persista.

**Contra essa leitura otimista há um contraponto.** Lesão corporal dolosa subiu 43,6% e tentativa de homicídio, 71,8% — ambas violências contra a pessoa, e a segunda é justamente a natureza que menos depende de propensão a registrar (dificilmente uma tentativa de homicídio deixa de virar boletim). Ou seja: a violência patrimonial caiu, mas a violência interpessoal cresceu. São fenômenos diferentes, com causas diferentes, e o número agregado da P1 esconde os dois.

**Esta é a resposta que justifica o modelo dimensional.** A hierarquia categoria → natureza da `dim_natureza` é o que permite subir e descer entre "criminalidade cresceu 4,9%" e "roubo caiu 33% enquanto lesão corporal subiu 44%" com a mesma consulta e o mesmo dado.

---
## P3. Existe sazonalidade mensal? O padrão se repete entre os anos?

Esta pergunta usa a **data da ocorrência**, e não a da estatística: sazonalidade é propriedade de quando o fato acontece.

In [ ]:
p3 = consultar("""
SELECT
  mes_ocorrencia,
  ANY_VALUE(nome_mes_ocorrencia) AS mes,
  SUM(IF(ano_ocorrencia = 2022, qtd_ocorrencia, 0)) AS ano_2022,
  SUM(IF(ano_ocorrencia = 2023, qtd_ocorrencia, 0)) AS ano_2023,
  SUM(IF(ano_ocorrencia = 2024, qtd_ocorrencia, 0)) AS ano_2024,
  SUM(IF(ano_ocorrencia = 2025, qtd_ocorrencia, 0)) AS ano_2025,
  SUM(qtd_ocorrencia)                               AS total
FROM `@projeto.dw.vw_ocorrencias`
WHERE ano_ocorrencia BETWEEN 2022 AND 2025
GROUP BY mes_ocorrencia
ORDER BY mes_ocorrencia
""")
p3["indice"] = (100 * p3.total / p3.total.mean()).round(1)
display(p3)

figura, (esq, dir) = plt.subplots(1, 2, figsize=(13, 4))
for ano in ["ano_2022", "ano_2023", "ano_2024", "ano_2025"]:
    esq.plot(p3.mes.str[:3], p3[ano], marker="o", label=ano.replace("ano_", ""))
esq.legend(); esq.set_title("Ocorrências por mês, ano a ano")
cores = ["#c0392b" if v > 100 else "#7f8c8d" for v in p3.indice]
dir.bar(p3.mes.str[:3], p3.indice, color=cores)
dir.axhline(100, color="black", linewidth=0.8, linestyle="--")
dir.set_title("Índice sazonal (média do período = 100)")
dir.set_ylim(80, 112)
plt.tight_layout(); plt.show()

### Discussão

**Resposta:** existe sazonalidade, mas ela é **fraca**. A amplitude entre o mês de maior e o de menor volume é de 22% — agosto concentra 5.751 ocorrências e fevereiro, 4.715.

**E a maior parte dessa amplitude é artefato de calendário, não comportamento criminal.** Fevereiro tem 28 dias contra 31 de agosto: 10,7% de diferença apenas no número de dias. Corrigida essa distorção, a sazonalidade real cai para cerca de 10% entre extremos — um efeito pequeno.

**O padrão se repete entre os anos?** Parcialmente. Fevereiro é o mês mais baixo em todos os quatro anos, o que é consistente. Já o pico oscila: o segundo semestre tende a ser mais movimentado, com agosto e outubro acima da média, mas qual mês exatamente lidera muda de ano para ano.

**Conclusão prática:** não há um "mês do crime" em Sorocaba que justifique concentrar recursos sazonalmente. Para efeito de planejamento, o volume mensal é aproximadamente constante, e o pequeno excedente do segundo semestre não sustenta, sozinho, uma mudança de alocação. Esta é uma resposta negativa — e uma resposta negativa bem estabelecida é um resultado útil, porque evita decisões baseadas em um padrão que não existe.

---
## P4. Como as ocorrências se distribuem por período do dia e dia da semana, e isso muda conforme a natureza?

**Restrição declarada:** esta análise usa apenas os registros com hora informada — 75,3% do total. E a granularidade máxima confiável é a **hora cheia**: a análise de qualidade mostrou que 41% dos horários caem no minuto 00 e 17% no minuto 30, ou seja, são estimativas arredondadas, não medições.

In [ ]:
p4_periodo = consultar("""
SELECT
  periodo,
  SUM(qtd_ocorrencia) AS ocorrencias,
  ROUND(100 * SUM(qtd_ocorrencia) / SUM(SUM(qtd_ocorrencia)) OVER (), 1) AS percentual,
  ROUND(100 * SUM(IF(crime_violento, qtd_ocorrencia, 0)) / SUM(qtd_ocorrencia), 1) AS percentual_violento
FROM `@projeto.dw.vw_ocorrencias`
WHERE ano_estatistica BETWEEN 2022 AND 2025
GROUP BY periodo
ORDER BY ocorrencias DESC
""")
display(p4_periodo)

# Drill-down: do período para a hora cheia, comparando furto e roubo
p4_hora = consultar("""
SELECT
  hora,
  SUM(IF(natureza_apurada = 'FURTO - OUTROS', qtd_ocorrencia, 0)) AS furto,
  SUM(IF(natureza_apurada = 'ROUBO - OUTROS', qtd_ocorrencia, 0)) AS roubo
FROM `@projeto.dw.vw_ocorrencias`
WHERE hora_informada AND ano_estatistica BETWEEN 2022 AND 2025
GROUP BY hora
ORDER BY hora
""")
p4_hora["furto_%"] = (100 * p4_hora.furto / p4_hora.furto.sum()).round(1)
p4_hora["roubo_%"] = (100 * p4_hora.roubo / p4_hora.roubo.sum()).round(1)

figura, eixo = plt.subplots(figsize=(11, 4.5))
eixo.plot(p4_hora.hora, p4_hora["furto_%"], marker="o", label="Furto", color="#2980b9", linewidth=2)
eixo.plot(p4_hora.hora, p4_hora["roubo_%"], marker="s", label="Roubo", color="#c0392b", linewidth=2)
eixo.axvspan(18, 23, alpha=0.12, color="#c0392b")
eixo.set_xticks(range(0, 24))
eixo.set_xlabel("hora do dia"); eixo.set_ylabel("% das ocorrências da natureza")
eixo.set_title("Perfil horário: furto x roubo (2022-2025)")
eixo.legend()
plt.tight_layout(); plt.show()
display(p4_hora)

### Discussão

**Resposta:** os quatro períodos do dia são surpreendentemente equilibrados — madrugada 26,4%, noite 23,8%, tarde 23,0% e manhã 21,6%. Olhando só o agregado, a conclusão seria que **não há concentração horária** e que o policiamento deveria ser uniforme ao longo do dia.

**Essa conclusão é falsa, e o drill-down por natureza mostra por quê.** Quando a mesma medida é recortada por tipo de crime, dois perfis opostos aparecem:

| | Furto | Roubo |
|---|---|---|
| Pico | **3h da madrugada** (7,7%) | **5h e 21h** (7,2% e 6,8%) |
| Entre 18h e 23h | 19,2% | **36,2%** |

**O roubo concentra-se quase o dobro no período noturno.** Enquanto o furto se distribui de forma relativamente plana ao longo das 24 horas, o roubo — o crime que envolve confronto com a vítima — tem um pico vespertino-noturno claro a partir das 18h, exatamente o horário de retorno do trabalho.

**Consequência prática:** a alocação de policiamento ostensivo, cujo alvo é o crime com confronto, tem justificativa empírica para se concentrar entre 18h e 23h. Já a prevenção de furto não tem horário preferencial e depende de outros instrumentos.

**Uma ressalva sobre o pico de furto às 3h.** Ele deve ser lido com desconfiança: furtos são frequentemente **descobertos** muito depois de ocorridos, e a vítima informa uma hora estimada. O arredondamento massivo de horários (41% no minuto 00) reforça essa leitura. O perfil plano do furto é provavelmente mais real do que seu pico noturno aparente — enquanto o pico do roubo, crime presenciado pela vítima, é confiável.

---
## P5. Que tipos de local concentram cada natureza criminal?

**Restrição declarada:** esta análise usa apenas 2025 e 2026, os anos em que a fonte publica o tipo de local. Nos anos anteriores o tipo é derivado do subtipo, e 2,2% dessas derivações são ambíguas.

In [ ]:
p5 = consultar("""
SELECT
  tipo_local,
  SUM(qtd_ocorrencia) AS ocorrencias,
  ROUND(100 * SUM(qtd_ocorrencia) / SUM(SUM(qtd_ocorrencia)) OVER (), 1) AS percentual,
  ROUND(100 * SUM(IF(crime_violento, qtd_ocorrencia, 0)) / SUM(qtd_ocorrencia), 1) AS percentual_violento
FROM `@projeto.dw.vw_ocorrencias`
WHERE origem_tipo_local = 'publicado pela fonte'
GROUP BY tipo_local
HAVING ocorrencias >= 200
ORDER BY ocorrencias DESC
""")
display(p5)

figura, (esq, dir) = plt.subplots(1, 2, figsize=(13, 4.5))
esq.barh(p5.tipo_local[::-1], p5.ocorrencias[::-1], color="#34495e")
esq.set_title("Volume por tipo de local")
cores = ["#c0392b" if v > 25 else "#7f8c8d" for v in p5.percentual_violento[::-1]]
dir.barh(p5.tipo_local[::-1], p5.percentual_violento[::-1], color=cores)
dir.set_title("% de crimes violentos no local")
dir.set_xlabel("%")
plt.tight_layout(); plt.show()

### Discussão

**Resposta:** a criminalidade de Sorocaba é predominantemente **de rua**. A via pública concentra **55,4%** de todas as ocorrências, seguida por residência (18,1%). Nenhum outro tipo de local passa de 7%.

**Mas o volume e a gravidade apontam para lugares diferentes**, e é o cruzamento das duas medidas que dá a resposta útil:

| Tipo de local | Volume | % violentos |
|---|---|---|
| Via Pública | 55,4% | 20,1% |
| Residência | 18,1% | **41,9%** |
| Estabelecimento de Ensino | 1,0% | **51,2%** |
| Condomínio Residencial | 1,5% | 38,2% |
| Estacionamento/Garagem | 2,6% | 2,7% |

**A residência é o local mais perigoso da cidade, não a rua.** Embora responda por menos de um quinto das ocorrências, 41,9% delas são crimes violentos — mais que o dobro da proporção na via pública. O que acontece na rua é majoritariamente furto; o que acontece dentro de casa é majoritariamente violência contra a pessoa. É a assinatura da **violência doméstica**, que não aparece em nenhuma estatística agregada por volume.

**Estabelecimentos de ensino têm a maior proporção de crimes violentos de todas** — 51,2%. O volume é baixo (1% do total), mas mais da metade do que ali se registra envolve violência ou grave ameaça.

**Consequência prática:** políticas de segurança desenhadas a partir do volume alocariam quase tudo à via pública. A leitura por gravidade mostra que os locais que mais precisam de intervenção — residência e escola — são precisamente os que o policiamento ostensivo não alcança, e que demandam outros instrumentos: rede de proteção à mulher, mediação escolar, canais de denúncia.

---
## P6. Como as ocorrências se distribuem no território?

In [ ]:
p6_delegacia = consultar("""
SELECT
  delegacia,
  SUM(qtd_ocorrencia) AS ocorrencias,
  ROUND(100 * SUM(qtd_ocorrencia) / SUM(SUM(qtd_ocorrencia)) OVER (), 1) AS percentual,
  ROUND(100 * SUM(IF(crime_violento, qtd_ocorrencia, 0)) / SUM(qtd_ocorrencia), 1) AS percentual_violento
FROM `@projeto.dw.vw_ocorrencias`
WHERE ano_estatistica BETWEEN 2022 AND 2025
GROUP BY delegacia
ORDER BY ocorrencias DESC
""")
display(p6_delegacia)

figura, eixo = plt.subplots(figsize=(10, 4.5))
principais = p6_delegacia[p6_delegacia.ocorrencias >= 500]
posicoes = range(len(principais))
eixo.bar(posicoes, principais.ocorrencias, color="#34495e", label="ocorrências")
eixo.set_xticks(list(posicoes))
eixo.set_xticklabels(principais.delegacia, rotation=45, ha="right", fontsize=8)
eixo2 = eixo.twinx()
eixo2.plot(posicoes, principais.percentual_violento, color="#c0392b", marker="o", label="% violentos")
eixo2.set_ylabel("% de crimes violentos", color="#c0392b")
eixo2.grid(False)
eixo.set_title("Ocorrências e proporção de crimes violentos por delegacia de circunscrição")
plt.tight_layout(); plt.show()

In [ ]:
p6_bairro = consultar("""
SELECT
  nome_bairro,
  SUM(qtd_ocorrencia) AS ocorrencias,
  ROUND(100 * SUM(IF(crime_violento, qtd_ocorrencia, 0)) / SUM(qtd_ocorrencia), 1) AS percentual_violento
FROM `@projeto.dw.vw_ocorrencias`
WHERE bairro_informado AND ano_estatistica BETWEEN 2022 AND 2025
GROUP BY nome_bairro
ORDER BY ocorrencias DESC
LIMIT 15
""")
display(p6_bairro)

### Discussão

**Resposta:** a distribuição territorial é **desigual, mas não extrema**. O 8º Distrito Policial concentra 18,7% das ocorrências e o 7º, apenas 0,8% — uma razão de mais de vinte vezes entre o maior e o menor. As três maiores circunscrições (8º, 9º e 3º DP) somam **46,5%** de tudo.

**De novo, volume e gravidade não coincidem.** O 7º DP, com o menor volume da cidade, tem a **maior proporção de crimes violentos: 39,7%** — quase o dobro da média. O 6º DP (31,4%) e o 11º DP (31,0%) vêm em seguida. Já o 3º DP, terceiro em volume, tem a menor proporção de violência (17,9%).

Isso caracteriza dois tipos de território distintos, que uma leitura só por volume confundiria:

- **áreas de alto volume e baixa violência** — típicas de regiões centrais e comerciais, onde predomina o furto;
- **áreas de baixo volume e alta violência** — onde se registra menos, mas o que se registra é mais grave.

**Sobre o bairro.** O Centro lidera com 2.488 ocorrências (3,8% do total), seguido por Parque São Bento e Parque Campolim. A dispersão é grande: nenhum bairro passa de 4% e a cauda é longa, o que é coerente com uma cidade de mais de 700 mil habitantes sem uma única concentração dominante.

**Ressalva declarada.** Como a análise de qualidade mostrou, o campo bairro é de preenchimento livre e mantém, mesmo após padronização, cardinalidade acima da real. O ranking dos bairros de maior volume é confiável — grafias variantes de um mesmo bairro grande continuam somando volume suficiente para aparecer —, mas a cauda longa contém fragmentação. Por isso a resposta principal a esta pergunta usa a **delegacia de circunscrição**, campo controlado com 11 valores, e o bairro serve como detalhamento.

---
## P7. Furto e roubo de veículo: evolução e participação no total

In [ ]:
p7 = consultar("""
SELECT
  ano,
  SUM(IF(natureza_apurada = 'FURTO DE VEICULO', ocorrencias, 0))      AS furto_de_veiculo,
  SUM(IF(natureza_apurada = 'ROUBO DE VEICULO', ocorrencias, 0))      AS roubo_de_veiculo,
  ROUND(SUM(IF(natureza_apurada = 'FURTO DE VEICULO', taxa_por_100_mil, 0)), 1) AS taxa_furto,
  ROUND(SUM(IF(natureza_apurada = 'ROUBO DE VEICULO', taxa_por_100_mil, 0)), 1) AS taxa_roubo,
  ROUND(100 * SUM(IF(natureza_apurada IN ('FURTO DE VEICULO', 'ROUBO DE VEICULO'),
                     ocorrencias, 0)) / SUM(ocorrencias), 1) AS percentual_do_total
FROM `@projeto.dw.vw_taxa_anual`
GROUP BY ano
ORDER BY ano
""")
display(p7)

figura, eixo = plt.subplots(figsize=(9, 4.5))
eixo.plot(p7.ano.astype(str), p7.taxa_furto, marker="o", linewidth=2,
          label="Furto de veículo", color="#2980b9")
eixo.plot(p7.ano.astype(str), p7.taxa_roubo, marker="s", linewidth=2,
          label="Roubo de veículo", color="#c0392b")
eixo.set_ylabel("taxa por 100 mil habitantes")
eixo.set_title("Crimes contra veículos — taxa por 100 mil habitantes")
eixo.legend()
for x, y in zip(p7.ano.astype(str), p7.taxa_furto):
    eixo.text(x, y, f"{y:.0f}", ha="center", va="bottom", fontsize=9)
for x, y in zip(p7.ano.astype(str), p7.taxa_roubo):
    eixo.text(x, y, f"{y:.0f}", ha="center", va="top", fontsize=9)
plt.tight_layout(); plt.show()

### Discussão

**Resposta:** os crimes contra veículos respondem por cerca de **12% de todas as ocorrências** de Sorocaba e se moveram em direções opostas no período:

| | 2022 | 2025 | Variação da taxa |
|---|---|---|---|
| **Furto** de veículo | 177,6 / 100 mil | 230,5 / 100 mil | **+30%** |
| **Roubo** de veículo | 46,3 / 100 mil | 33,5 / 100 mil | **-28%** |

**A razão entre furto e roubo de veículo passou de 3,8 para 6,9 em quatro anos.** Em 2022, para cada roubo havia quase quatro furtos; em 2025, quase sete. Esta é a confirmação mais nítida do padrão já observado em P2: **o crime patrimonial em Sorocaba está migrando da modalidade violenta para a não violenta**.

**Por que este indicador é o mais confiável de todos.** Furto e roubo de veículo têm subnotificação baixíssima — o registro do boletim é exigido pela seguradora e para o bloqueio do documento. Enquanto o crescimento geral de P1 pode refletir maior propensão a registrar, aqui a propensão já era próxima do máximo em 2022. **A queda de 28% no roubo de veículo é, portanto, muito provavelmente uma queda real**, e não um artefato de registro.

**O contraponto.** O furto de veículo subiu 30% em taxa, e isso é um problema real: mais gente teve o carro levado. Mas menos gente foi abordada com violência para isso. Do ponto de vista da integridade física da população, é uma melhora; do ponto de vista patrimonial, uma piora.

---
## P8. A geolocalização permite identificar concentração espacial?

Esta era a **pergunta de risco** declarada no objetivo: sua viabilidade dependia inteiramente da qualidade de um campo que a fonte preenche de forma irregular.

In [ ]:
p8_cobertura = consultar("""
SELECT
  ano_estatistica AS ano,
  COUNT(*)                                               AS ocorrencias,
  COUNTIF(tem_geolocalizacao)                            AS com_coordenada,
  ROUND(100 * COUNTIF(tem_geolocalizacao) / COUNT(*), 1) AS percentual
FROM `@projeto.dw.vw_ocorrencias`
GROUP BY ano
ORDER BY ano
""")
display(p8_cobertura)

# Concentração: células de ~110 x 100 metros
p8_celulas = consultar("""
SELECT
  ROUND(latitude, 3)  AS latitude,
  ROUND(longitude, 3) AS longitude,
  ANY_VALUE(nome_bairro) AS bairro_predominante,
  SUM(qtd_ocorrencia) AS ocorrencias,
  ROUND(100 * SUM(IF(crime_violento, qtd_ocorrencia, 0)) / SUM(qtd_ocorrencia), 1) AS percentual_violento
FROM `@projeto.dw.vw_ocorrencias`
WHERE tem_geolocalizacao
  AND latitude BETWEEN -23.60 AND -23.35
  AND longitude BETWEEN -47.60 AND -47.35
  AND ano_estatistica BETWEEN 2022 AND 2025
GROUP BY latitude, longitude
ORDER BY ocorrencias DESC
""")
print(f"células de ~110 metros com ao menos uma ocorrência: {len(p8_celulas):,}")
display(p8_celulas.head(10))

In [ ]:
# Mapa de dispersão das ocorrências geolocalizadas
figura, eixo = plt.subplots(figsize=(7.5, 7))
dispersao = eixo.scatter(p8_celulas.longitude, p8_celulas.latitude,
                         s=p8_celulas.ocorrencias / 4,
                         c=p8_celulas.ocorrencias, cmap="YlOrRd",
                         alpha=0.65, edgecolors="none")
eixo.set_xlabel("longitude"); eixo.set_ylabel("latitude")
eixo.set_title("Concentração espacial das ocorrências em Sorocaba (2022-2025)\n"
               "tamanho e cor proporcionais ao número de ocorrências na célula")
plt.colorbar(dispersao, label="ocorrências na célula")
plt.tight_layout(); plt.show()

# Quanto do total as células mais densas concentram
p8_celulas["acumulado_%"] = (100 * p8_celulas.ocorrencias.cumsum()
                             / p8_celulas.ocorrencias.sum()).round(1)
print(f"as 10 células mais densas concentram {p8_celulas['acumulado_%'].iloc[9]}% das ocorrências geolocalizadas")
print(f"as 100 células mais densas concentram {p8_celulas['acumulado_%'].iloc[99]}%")

### Discussão

**Resposta: sim, a pergunta de risco deu certo** — com uma ressalva importante.

**Viabilidade.** **61,5%** das ocorrências têm coordenada válida, e a cobertura é estável ao longo dos anos (de 59,1% a 66,0%), o que é decisivo: se a cobertura variasse muito entre anos, comparações temporais seriam impossíveis. Melhor ainda, **99,2% das coordenadas presentes caem dentro da moldura geográfica de Sorocaba** — as coordenadas que existem são confiáveis.

**Concentração.** Agregando em células de aproximadamente 110 metros, as ocorrências se espalham por cerca de 6.800 células. A célula mais densa, no **Centro**, acumula 430 ocorrências em quatro anos. As dez células mais densas concentram cerca de **5% de tudo** — uma concentração real, mas longe da regra empírica de "poucos pontos concentram a maioria dos crimes" observada em outras cidades.

**A leitura correta é de uma criminalidade dispersa com pontos quentes moderados.** Sorocaba não tem uma "cracolândia" estatística: tem um centro movimentado e alguns aglomerados em bairros como Aparecidinha, Vila Gagliari e Vila Pedro Ribeiro, mas o grosso das ocorrências se distribui por todo o tecido urbano. Isso tem consequência direta de política pública: estratégias de *hot spot policing*, que funcionam onde o crime é hiperconcentrado, teriam alcance limitado aqui.

**A ressalva que impede uma conclusão mais forte.** Os 38,5% sem coordenada **não são uma amostra aleatória**. É plausível que registros feitos pela Delegacia Eletrônica — que dominam o total — tenham geolocalização com qualidade diferente dos lavrados presencialmente. Sem saber o que falta, não é possível afirmar que o mapa dos 61,5% é o mapa da cidade. A conclusão sobre dispersão é sólida; a localização exata de cada ponto quente merece verificação antes de virar decisão operacional.

---

# Discussão geral: o problema foi resolvido?

O problema declarado no objetivo era transformar dados brutos e fragmentados do estado inteiro em uma base confiável e consultável sobre Sorocaba, capaz de sustentar respostas sobre **quando, onde e de que tipo** são as ocorrências criminais da cidade.

**Sobre o produto.** Os 5,3 milhões de registros do estado, distribuídos em cinco arquivos de 200 MB com esquemas incompatíveis entre si, tornaram-se um data warehouse dimensional com 73.394 fatos, sete dimensões e dois fatos conformados, no qual cada uma das oito perguntas é respondida por uma consulta de poucas linhas. As sete perguntas planejadas foram respondidas, e a oitava — declarada como aposta de risco — também.

**Sobre a criminalidade em Sorocaba, o que os dados dizem:**

1. **O volume cresceu menos do que parece.** 10,5% em números absolutos, mas 4,9% em taxa por 100 mil habitantes. Metade do crescimento aparente é crescimento populacional. O pico do período foi 2023, não 2025.

2. **A composição do crime mudou mais do que o volume.** Roubo caiu 33%, roubo de veículo caiu 24%, furto de veículo subiu 37%. A cidade registra hoje **menos crime patrimonial violento e mais crime patrimonial sem confronto** — e o indicador mais confiável da base (veículos, de subnotificação baixa) confirma o movimento.

3. **Mas a violência interpessoal cresceu.** Lesão corporal dolosa subiu 44% e tentativa de homicídio, 72%. São fenômenos distintos do crime patrimonial, e o número agregado esconde os dois.

4. **O lugar mais violento da cidade é dentro de casa.** A via pública tem o volume (55%), mas a residência tem a gravidade: 41,9% do que se registra ali é crime violento, contra 20,1% na rua. É a assinatura da violência doméstica, invisível em qualquer leitura por volume.

5. **O risco de confronto tem hora e a prevenção de furto não tem.** O roubo concentra 36,2% das ocorrências entre 18h e 23h; o furto se distribui de forma plana pelas 24 horas.

6. **O crime é territorialmente disperso.** Não há hiperconcentração: as dez células mais densas somam 5% do total. E as áreas de maior volume não são as de maior gravidade — o distrito com menos ocorrências da cidade é o de maior proporção de crimes violentos.

**A conclusão que atravessa todas as respostas** é que *criminalidade* não é uma grandeza única. Cada vez que uma pergunta foi respondida no agregado e depois recortada por natureza, por local ou por horário, o recorte contradisse o agregado. É exatamente para isso que serve um modelo dimensional: permitir que a mesma medida seja vista sob perspectivas diferentes com a mesma consulta — e é o que justifica o custo de construir o warehouse em vez de responder cada pergunta com um script isolado sobre a planilha.

**O que a base não permite concluir.** Ela mede ocorrências **comunicadas**, não crimes ocorridos. Nenhuma das variações acima pode ser atribuída com certeza a mudança na criminalidade em vez de mudança na propensão a registrar — exceto os crimes contra veículos, cuja subnotificação já era mínima. Essa limitação é da fonte, não do trabalho, e está declarada desde o objetivo.

A discussão sobre o atingimento dos objetivos, as dificuldades encontradas e o trabalho futuro está em [`docs/08-autoavaliacao.md`](../docs/08-autoavaliacao.md).